In [23]:
import sys, subprocess, pkgutil

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg,
                               "--break-system-packages", "-q"])

ensure("openpyxl")
ensure("statsmodels")
ensure("scipy")
ensure("matplotlib")
ensure("pandas")
ensure("numpy")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patheffects import withStroke
from scipy.stats import ttest_ind_from_stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations

# ── SETTINGS ──────────────────────────────────────────────────────────────────
xlsx_path  = "baselines.xlsx"
sheet_name = 0

n_per_dataset = {"D1": 331, "D2": 105, "D3": 106, "D4": 94}

block_specs = [
    ("Pigment (no spatial augmentation)", 0),
    ("Pigment", 12),
    ("MicroSAM", 24),
    ("UniVerSeg", 36),
]

datasets        = ["D1", "D2", "D3", "D4"]
metrics_to_plot = ["Dice", "IoU", "Det"]

methods_order = [
    "MicroSAM",
    "UniVerSeg",
    "Pigment (no spatial augmentation)",
    "Pigment",
]

colors = {
    "MicroSAM":                          "#B07070",
    "UniVerSeg":                          "#B89A50",
    "Pigment (no spatial augmentation)": "#6AAED6",
    "Pigment":                            "#1A5FA8",
}

hatches = {
    "MicroSAM":                          "////",
    "UniVerSeg":                          "xxxx",
    "Pigment (no spatial augmentation)": "",
    "Pigment":                            "",
}

short_labels = {
    "MicroSAM":                          "MicroSAM",
    "UniVerSeg":                          "UniVerSeg",
    "Pigment (no spatial augmentation)": "Pigment (Intensity Aug.)",
    "Pigment":                            "Pigment (Geometric Aug.)",
}

target_configs = [
    "D1",
    "D1 + (D2+D3)",
    "D1 + (D2+D3) + D4",
    "D1 + D4",
]

reference_method = "Pigment"
noaug_method     = "Pigment (no spatial augmentation)"
baselines        = ["MicroSAM", "UniVerSeg"]
sig_alpha        = 0.05

# ── LOAD ──────────────────────────────────────────────────────────────────────
raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, engine="openpyxl")

# ── HELPERS ───────────────────────────────────────────────────────────────────
def clean_cfg(x):
    if pd.isna(x): return None
    s = str(x).strip().replace(" ", "")
    mapping = {
        "D1": "D1",
        "D1+(D2+D3)": "D1 + (D2+D3)",
        "D1+(D2+D3)+D4": "D1 + (D2+D3) + D4",
        "D1+D4": "D1 + D4",
        "D1[N=128]": "D1",
        "D1[N=64]": "D1",
        "D1+D4[N=188]": "D1 + D4",
        "D1+D2+D3[N=188]": "D1 + (D2+D3)",
        "D1+D2+D3+D4[N=188]": "D1 + (D2+D3) + D4",
    }
    return mapping.get(s, s)

def fmt_label(v):
    if pd.isna(v):    return ""
    if v == 0:        return "0.00"
    if abs(v) < 0.05: return "<.05"
    return f"{v:.2f}"

def p_to_star(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

def fname(metric):
    return metric.lower().replace(" ", "_")

def extract_block(raw_df, start_col, method_name, metric_name):
    rows = []
    current_cfg = None
    for i in range(raw_df.shape[0]):
        c0 = raw_df.iloc[i, start_col]
        c1 = raw_df.iloc[i, start_col + 1]
        c2 = raw_df.iloc[i, start_col + 2]
        c0s = "" if pd.isna(c0) else str(c0).strip()
        c1s = "" if pd.isna(c1) else str(c1).strip()
        c2s = "" if pd.isna(c2) else str(c2).strip()

        if c0s.lower() == "train" and c1s:
            current_cfg = clean_cfg(c1s)
        if method_name == "MicroSAM" and c0s.lower() == "base" and c1s:
            current_cfg = "ALL_CONFIGS"
        if method_name == "UniVerSeg" and c0s.lower() in ["max n", "max"] and c1s:
            pool_map = {
                "D1": "D1", "D1+D4": "D1 + D4",
                "D1+D2+D3": "D1 + (D2+D3)",
                "D1+D2+D3+D4": "D1 + (D2+D3) + D4",
            }
            current_cfg = pool_map.get(clean_cfg(c1s), clean_cfg(c1s))

        if c2s == metric_name and current_cfg is not None:
            row = {"Method": method_name, "TrainConfig": current_cfg}
            for j, d in enumerate(datasets):
                row[f"{d}_mean"] = pd.to_numeric(raw_df.iloc[i, start_col + 3 + 2*j], errors="coerce")
                row[f"{d}_std"]  = pd.to_numeric(raw_df.iloc[i, start_col + 4 + 2*j], errors="coerce")
            rows.append(row)

    out = pd.DataFrame(rows)
    if method_name == "MicroSAM" and len(out):
        expanded = []
        for _, r in out.iterrows():
            if r["TrainConfig"] == "ALL_CONFIGS":
                for cfg in target_configs:
                    rr = r.copy(); rr["TrainConfig"] = cfg; expanded.append(rr)
            else:
                expanded.append(r)
        out = pd.DataFrame(expanded)
    return out

def build_metric_df(metric_name):
    df = pd.concat(
        [extract_block(raw, sc, nm, metric_name) for nm, sc in block_specs],
        ignore_index=True,
    )
    df = df[df["TrainConfig"].isin(target_configs)].copy()
    return df.drop_duplicates(subset=["Method", "TrainConfig"], keep="first").reset_index(drop=True)

def run_tests(all_df):
    tests = []
    for cfg in target_configs:
        sub = all_df[all_df["TrainConfig"] == cfg]
        avail = [m for m in methods_order if m in sub["Method"].values]
        for d in datasets:
            for m1, m2 in combinations(avail, 2):
                r1 = sub[sub["Method"] == m1].iloc[0]
                r2 = sub[sub["Method"] == m2].iloc[0]
                v = [r1[f"{d}_mean"], r1[f"{d}_std"], r2[f"{d}_mean"], r2[f"{d}_std"]]
                if any(pd.isna(x) for x in v): continue
                _, p = ttest_ind_from_stats(
                    mean1=v[0], std1=v[1], nobs1=n_per_dataset[d],
                    mean2=v[2], std2=v[3], nobs2=n_per_dataset[d],
                    equal_var=False,
                )
                tests.append({"TrainConfig": cfg, "Dataset": d,
                               "Method1": m1, "Method2": m2, "p_raw": p})
    tests_df = pd.DataFrame(tests)
    if len(tests_df):
        corrected = []
        for (cfg, d), grp in tests_df.groupby(["TrainConfig", "Dataset"], sort=False):
            _, p_adj, _, _ = multipletests(grp["p_raw"].values, alpha=sig_alpha, method="fdr_bh")
            tmp = grp.copy()
            tmp["p_adj"] = p_adj
            tmp["stars"] = [p_to_star(p) for p in p_adj]
            corrected.append(tmp)
        tests_df = pd.concat(corrected, ignore_index=True)
    return tests_df

# ── PLOT ──────────────────────────────────────────────────────────────────────
def plot_metric(metric_name):
    all_df   = build_metric_df(metric_name)
    tests_df = run_tests(all_df)

    plt.rcParams.update({
        "font.family": "sans-serif",
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    })

    fig, axes = plt.subplots(2, 2, figsize=(17, 12), sharey=True)
    axes = axes.flatten()

    n_m  = len(methods_order)
    tw   = 0.74
    w    = tw / n_m
    offs = np.array([i * w - tw/2 + w/2 for i in range(n_m)])
    omap = dict(zip(methods_order, offs))
    x    = np.arange(len(datasets))

    gmax = 0.0
    for _, row in all_df.iterrows():
        for d in datasets:
            v, e = row[f"{d}_mean"], row[f"{d}_std"]
            if pd.notna(v) and pd.notna(e):
                gmax = max(gmax, v + e)

    for ax, cfg in zip(axes, target_configs):
        sub = all_df[all_df["TrainConfig"] == cfg].copy()

        # subtle tint behind Pigment columns
        for di in range(len(datasets)):
            sx = x[di] + offs[2] - w * 0.46
            sw = w * 2 + w * 0.08
            ax.axvspan(sx, sx + sw, color="#1A5FA8", alpha=0.04, zorder=1)

        # dashed divider between baselines and Pigment
        for di in range(len(datasets)):
            ax.axvline(x[di] + offs[1] + w / 2,
                       color="#bbbbbb", lw=0.7, ls=(0, (4, 3)), zorder=2)

        # bars
        for method in methods_order:
            rr = sub[sub["Method"] == method]
            if rr.empty: continue
            rr = rr.iloc[0]
            means = [rr[f"{d}_mean"] for d in datasets]
            errs  = [rr[f"{d}_std"]  for d in datasets]

            # clip lower error bar so it never goes below 0
            err_lo = [min(e, v) if pd.notna(e) and pd.notna(v) else e
                      for v, e in zip(means, errs)]
            err_hi = [e if pd.notna(e) else 0 for e in errs]
            yerr_asym = [err_lo, err_hi]

            ax.bar(
                x + omap[method], means, w * 0.86,
                yerr=yerr_asym, capsize=2.5,
                color=colors[method], hatch=hatches[method],
                edgecolor="white", linewidth=0.4,
                error_kw=dict(elinewidth=0.9, capthick=0.9, ecolor="#444444"),
                alpha=0.92, zorder=3,
            )

            # value labels with white stroke outline (no box)
            for di, (val, err) in enumerate(zip(means, errs)):
                if pd.isna(val): continue
                top = val + (err if pd.notna(err) else 0)
                ax.text(
                    x[di] + omap[method], top + 0.010, fmt_label(val),
                    ha="center", va="bottom",
                    fontsize=7, fontweight="bold", color="#111111",
                    clip_on=False,
                    path_effects=[withStroke(linewidth=2.5, foreground="white")],
                )

        # significance brackets
        if len(tests_df):
            ref_tests = tests_df[
                (tests_df["TrainConfig"] == cfg) &
                (
                    (tests_df["Method1"] == reference_method) |
                    (tests_df["Method2"] == reference_method)
                ) &
                (tests_df["stars"] != "")
            ].copy()

            def other(row):
                return row["Method2"] if row["Method1"] == reference_method else row["Method1"]
            ref_tests["Other"] = ref_tests.apply(other, axis=1)

            for di, d in enumerate(datasets):
                dd = ref_tests[ref_tests["Dataset"] == d]

                b_noaug    = dd[dd["Other"] == noaug_method].sort_values("p_adj")
                b_baseline = dd[dd["Other"].isin(baselines)].sort_values("p_adj").head(1)

                brackets = []
                if not b_noaug.empty:    brackets.append(b_noaug.iloc[0])
                if not b_baseline.empty: brackets.append(b_baseline.iloc[0])

                # y anchor = top of tallest bar+error in this dataset
                tops = []
                for m in methods_order:
                    rr = sub[sub["Method"] == m]
                    if rr.empty: continue
                    v, e = rr.iloc[0][f"{d}_mean"], rr.iloc[0][f"{d}_std"]
                    if pd.notna(v) and pd.notna(e): tops.append(v + e)
                y_anchor = max(tops) if tops else 0.0

                for level, row in enumerate(brackets):
                    m1, m2, star = row["Method1"], row["Method2"], row["stars"]
                    x1 = x[di] + omap[m1]
                    x2 = x[di] + omap[m2]
                    if x1 > x2: x1, x2 = x2, x1

                    y0 = y_anchor + 0.048 + level * 0.060
                    th = 0.011

                    ax.plot([x1, x1, x2, x2],
                            [y0, y0 + th, y0 + th, y0],
                            color="#222222", lw=1.0, solid_capstyle="round")
                    ax.text((x1 + x2) / 2, y0 + th + 0.003, star,
                            ha="center", va="bottom",
                            fontsize=8.5, fontweight="bold", color="#222222")

        ax.set_title(cfg, fontsize=11, fontweight="bold", pad=5)
        ax.set_xticks(x)
        ax.set_xticklabels(datasets)
        ax.set_xlabel("Test dataset", labelpad=3, fontsize=13)
        ax.set_ylabel(metric_name, fontsize=13)
        ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.35, zorder=0)
        ax.set_axisbelow(True)
        for spine in ["top", "right"]: ax.spines[spine].set_visible(False)
        ax.spines["left"].set_linewidth(0.5)
        ax.spines["bottom"].set_linewidth(0.5)

    for ax in axes:
        ax.set_ylim(0.0, gmax + 0.32)

    # legend — Nature-style: boxed, top-right of last subplot
    patches = [
        mpatches.Patch(
            facecolor=colors[m], hatch=hatches[m],
            edgecolor="#999" if hatches[m] else colors[m],
            linewidth=0.5, label=short_labels[m],
        )
        for m in methods_order
    ]
    axes[1].legend(
        handles=patches, loc="upper right",
        frameon=True, fontsize=11,
        handlelength=1.6, handleheight=1.1,
        borderpad=0.7, labelspacing=0.5, handletextpad=0.6,
        framealpha=1.0,
        edgecolor="#aaaaaa",
        fancybox=False,
    )

    fig.suptitle(
        f"{metric_name} — comparison across training configurations",
        fontsize=14, fontweight="bold", y=1.01,
    )

    plt.tight_layout(h_pad=3.2, w_pad=2.5)

    png = f"{fname(metric_name)}_comparison.png"
    pdf = f"{fname(metric_name)}_comparison.pdf"
    plt.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.savefig(pdf,           bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {png}  |  {pdf}")

# ── RUN ───────────────────────────────────────────────────────────────────────
for m in metrics_to_plot:
    plot_metric(m)

Saved: dice_comparison.png  |  dice_comparison.pdf
Saved: iou_comparison.png  |  iou_comparison.pdf
Saved: det_comparison.png  |  det_comparison.pdf
